In [ ]:
# %% Notebook unificado: setup, estilos, carga de dados e exemplo de uso
# Mantém a lógica original, mas centraliza estilos e comentários em uma única célula.

# -- Imports e setup ----------------------------------------------
from __future__ import annotations
from pathlib import Path
import sys
import warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

# Descobre a raiz do repositório (funciona se o notebook estiver em Códigos Extras/)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'project').exists():
    candidatos = [p for p in (REPO_ROOT.parent, REPO_ROOT.parent.parent) if (p / 'project').exists()]
    if candidatos:
        REPO_ROOT = candidatos[0]

SRC_ROOT = REPO_ROOT / 'project' / 'scripts'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from src.core.config import configure_runtime, validate_data_files, DATA_DIR
from src.core import repo_global as repo
from src.data.datasets import load_all
from src.domain.builders import build_dataframe_diario_area, build_dataframe_almoco_area
from src.domain.periods import periodo_limites

# Aplica locale/pandas e valida existência dos .parquet essenciais
configure_runtime()
validate_data_files()

# -- Estilos reutilizáveis ----------------------------------------
# Paletas e presets centralizados para gráficos e tabelas (DRY).
STYLE_COLORS = {
    'background': '#ffffff',
    'text': '#111111',
    'primary': '#0b72b9',
    'accent': '#ef6464',
    'neutral': '#e0e0e0',
}

STYLE_TABLE = {
    'header': {'backgroundColor': '#f2f2f2', 'fontWeight': 'bold'},
    'cell': {'padding': '6px 10px', 'border': '1px solid #e5e5e5'},
}

STYLE_PLOTLY = {
    'layout': {
        'paper_bgcolor': 'white',
        'plot_bgcolor': 'white',
        'font': {'family': 'Arial', 'size': 12, 'color': STYLE_COLORS['text']},
        'margin': {'l': 60, 'r': 40, 't': 60, 'b': 50},
    },
    'line': {'color': STYLE_COLORS['primary'], 'width': 2},
    'bar': {'marker': {'color': STYLE_COLORS['primary'], 'line': {'width': 0}}},
}
# Use STYLE_* para aplicar estilos em gráficos/tabelas sem repetir dicionários.

# -- Carregamento das bases ---------------------------------------
data = load_all()

df_semana = data['df_semana_merged']
df_datas = data['df_datas']
df_departamentos = data['df_departamentos']
df_afastados = data['df_afastados']
df_gptw = data['df_gptw']
df_almoco = data['df_almoco']

print(f'Dados carregados de: {DATA_DIR}')
for nome, df in data.items():
    print(f'- {nome}: {df.shape[0]} linhas | {df.shape[1]} colunas')

# -- Utilitários (comentados) ------------------------------------
def describe_frame(df: pd.DataFrame, titulo: str, max_rows: int = 5) -> None:
    """Exibe estatísticas rápidas de um DataFrame (head limitado).

    Args:
        df: DataFrame a ser inspecionado.
        titulo: rótulo exibido no print.
        max_rows: número de linhas a exibir na amostra.
    """
    print(f'[{titulo}] linhas={len(df)} colunas={list(df.columns)}')
    display(df.head(max_rows))

def recorte_presenca(area: str | None = 'GERAL', subarea: str | None = None, periodo_idx: tuple[int, int] | None = None):
    """Gera recortes de presença e almoço por área/subárea usando funções do domínio.

    Args:
        area: slug de área (ou 'GERAL' para todas).
        subarea: subárea específica opcional.
        periodo_idx: tupla de índices do slider (ex.: (0, 5)) para limitar datas.
    Returns:
        dict: DataFrames 'presenca' e 'almoco' prontos para análise/visualização.
    """
    dt_ini = dt_fim = None
    if periodo_idx is not None:
        dt_ini, dt_fim = periodo_limites(periodo_idx)

    diario = build_dataframe_diario_area(
        area=area,
        subarea=subarea,
        her_df=repo.her,
        df_datas_local=df_datas,
        df_dept=df_departamentos,
        dt_ini=dt_ini,
        dt_fim=dt_fim,
    )

    almoco_df = build_dataframe_almoco_area(
        area=area,
        subarea=subarea,
        her_df=repo.her,
        df_almoco_local=df_almoco,
        df_dept=df_departamentos,
        dt_ini=dt_ini,
        dt_fim=dt_fim,
    )

    return {'presenca': diario, 'almoco': almoco_df}

# -- Exemplo rápido ----------------------------------------------
area_exemplo = 'GERAL'
subarea_exemplo = None
periodo_exemplo = None  # use tupla (i0, i1) se quiser limitar datas

amostras = recorte_presenca(area=area_exemplo, subarea=subarea_exemplo, periodo_idx=periodo_exemplo)
describe_frame(amostras['presenca'], 'Presença (diário)')
describe_frame(amostras['almoco'], 'Almoço (janela de 11h-14h)')
